# 02 — OpenSurvey EDA + 외부 검증 (Phase 1.1 + 1.2)

## 목적
한국 카페 시장 인구통계 × 메뉴/카페/시간대 prior를 추출하고, 외부 reference와 정량 비교해 합성 데이터의 ground truth를 확정한다.

## 입력
- `kr_synthetic/opensurvey_cafe_2025_synthetic.csv` (2,000명 × 148문항)
- `kr_synthetic/rec_users.csv` (사용자 인구통계 마스터)
- `kr_synthetic/rec_items.csv` (한국 카페 메뉴 30개 메타)
- 검증 reference (옵션):
  - Bread Basket EDA 결과 (`output/transactions_clean.csv` — 노트북 01 산출)
  - Coffee Sales Dataset (`coffee sales dataset/Coffe_sales.csv`)

## 출력
- `output/prior_demographics.csv`            P(gender, age_band, job, area)
- `output/prior_menu_by_demographic.csv`     P(menu_category | gender, age_band)
- `output/prior_brand_by_demographic.csv`    P(cafe_brand | demographic)
- `output/prior_time_by_demographic.csv`     P(time_period | demographic)
- `output/prior_menu_by_time.csv`            P(menu | time_period)
- `output/validation_report.md`              외부 reference 검증 결과

## 데이터 업로드
**옵션 A — 직접 업로드**: 좌측 패널에서 `kr_synthetic/` 폴더 통째로 압축 후 업로드 → 압축 풀기.
**옵션 B — Drive 마운트** (권장): Drive에 `Adaptive_Kiosk/create_data/raw/kr_synthetic/`로 폴더 둠.

본 노트북 0번 셀에서 `USE_GDRIVE` 토글로 선택.

## 0. Imports & Paths

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

USE_GDRIVE = False

if USE_GDRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/Adaptive_Kiosk/create_data")
else:
    DATA_DIR = Path("/content")

RAW_DIR = DATA_DIR / "raw" / "kr_synthetic"
OUTPUT_DIR = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OPENSURVEY_PATH = RAW_DIR / "opensurvey_cafe_2025_synthetic.csv"
USERS_PATH      = RAW_DIR / "rec_users.csv"
ITEMS_PATH      = RAW_DIR / "rec_items.csv"

print("DATA_DIR :", DATA_DIR)
print("RAW_DIR  :", RAW_DIR)
print("OUTPUT   :", OUTPUT_DIR)
for p in (OPENSURVEY_PATH, USERS_PATH, ITEMS_PATH):
    print(f"  {p.name:50s} exists={p.exists()}")

## 1. Load (한글 인코딩 주의)

OpenSurvey CSV는 cp949(또는 euc-kr) 추정. 실패 시 utf-8 / utf-8-sig 순으로 시도.

In [ ]:
def read_csv_smart(path: Path) -> pd.DataFrame:
    last_err = None
    for enc in ("cp949", "euc-kr", "utf-8-sig", "utf-8"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as e:
            last_err = e
            continue
    raise RuntimeError(f"Cannot read {path}: {last_err}")

survey_df = read_csv_smart(OPENSURVEY_PATH)
users_df  = read_csv_smart(USERS_PATH)
items_df  = read_csv_smart(ITEMS_PATH)

print("survey:", survey_df.shape)
print("users :", users_df.shape)
print("items :", items_df.shape)
survey_df.head(2)

In [ ]:
print("[users] columns:", list(users_df.columns))
print("\n[items] columns:", list(items_df.columns))
print("\n[survey] columns count:", len(survey_df.columns))
print("survey columns sample:", list(survey_df.columns[:30]))

## 2. 컬럼 그룹화

OpenSurvey는 컬럼 패턴이 복잡 — 그룹별로 나눠 분석한다.

- 인구통계: `respondent_id`, `sex`, `age`, `age_10`, `age_5`, `job_group`, `job`, `area`, `area_group`, `area_capital`
- SQ3_*: 평소 자주 마시는 음료 (binary multi-select)
- SQ4_1/2/3순위: 자주 가는 카페 브랜드 (single)
- visit_<카페>: 그 카페 방문 여부 (binary)
- <카페>_Q9_<메뉴>: 그 카페에서 마시는 메뉴 (binary)
- <카페>_Q8_<시간대>: 그 카페 방문 시간대 (binary)

In [ ]:
demo_cols   = [c for c in survey_df.columns
               if c in {"respondent_id","sex","age","age_10","age_5",
                         "job_group","job","area","area_group","area_capital"}]
sq3_cols    = [c for c in survey_df.columns if c.startswith("SQ3_")]
sq4_cols    = [c for c in survey_df.columns if c.startswith("SQ4_")]
visit_cols  = [c for c in survey_df.columns if c.startswith("visit_")]
q9_cols     = [c for c in survey_df.columns if "_Q9_" in c]
q8_cols     = [c for c in survey_df.columns if "_Q8_" in c]

print(f"demo: {len(demo_cols)}  sq3: {len(sq3_cols)}  sq4: {len(sq4_cols)}"
      f"  visit: {len(visit_cols)}  Q9: {len(q9_cols)}  Q8: {len(q8_cols)}")

# 카페 브랜드 / 메뉴 / 시간대 추출 (Q9·Q8 컬럼명 분해)
def split_q(col, marker):
    if marker not in col:
        return None, None
    brand, rest = col.split(marker, 1)
    return brand, rest

brands_q9   = sorted({split_q(c, '_Q9_')[0] for c in q9_cols})
menus_q9    = sorted({split_q(c, '_Q9_')[1] for c in q9_cols})
brands_q8   = sorted({split_q(c, '_Q8_')[0] for c in q8_cols})
times_q8    = sorted({split_q(c, '_Q8_')[1] for c in q8_cols})

print("\n[brands] Q9:", brands_q9)
print("[menus]  Q9:", menus_q9)
print("[times]  Q8:", times_q8)

## 3. 인구통계 분포 — `prior_demographics.csv`

In [ ]:
for col in ["sex", "age_10", "job_group", "area_group"]:
    print(f"\n[{col}]")
    print(survey_df[col].value_counts(dropna=False).to_string())

In [ ]:
demo_dist = (
    survey_df.groupby(["sex", "age_10", "job_group", "area_group"])
    .size().reset_index(name="count")
)
demo_dist["prob"] = demo_dist["count"] / demo_dist["count"].sum()
demo_dist.to_csv(OUTPUT_DIR / "prior_demographics.csv", index=False, encoding="utf-8-sig")
print("saved prior_demographics.csv shape:", demo_dist.shape)
demo_dist.head(10)

## 4. 평소 마시는 음료 (SQ3) × 인구통계 → `prior_menu_by_demographic.csv`

SQ3 컬럼은 binary multi-select. 각 메뉴별로 (sex × age_10) 그룹의 응답률 평균 = 그 인구의 메뉴 선호도.

In [ ]:
# 전체 SQ3 응답률
sq3_total = survey_df[sq3_cols].mean().sort_values(ascending=False)
print("[SQ3] 전체 응답률 (평소 마시는 음료):")
print(sq3_total.to_string())

fig, ax = plt.subplots(figsize=(8, 6))
sq3_total[::-1].plot(kind="barh", ax=ax, color="#0f172a")
ax.set_xlabel("response rate (mean)")
ax.set_title("SQ3 — 평소 마시는 음료 응답률 (전체)")
plt.tight_layout()
plt.show()

In [ ]:
# (sex × age_10) 별 SQ3 평균
menu_by_demo = (
    survey_df.groupby(["sex", "age_10"])[sq3_cols]
    .mean().reset_index()
)
menu_by_demo.to_csv(OUTPUT_DIR / "prior_menu_by_demographic.csv", index=False, encoding="utf-8-sig")
print("saved prior_menu_by_demographic.csv shape:", menu_by_demo.shape)
menu_by_demo.head()

In [ ]:
# 히트맵
pivot = menu_by_demo.set_index(["sex","age_10"])[sq3_cols]
fig, ax = plt.subplots(figsize=(min(len(sq3_cols)*0.5, 16), max(pivot.shape[0]*0.4, 4)))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(sq3_cols)))
ax.set_xticklabels([c.replace("SQ3_","") for c in sq3_cols], rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(pivot.shape[0]))
ax.set_yticklabels([f"{a}/{b}" for a,b in pivot.index], fontsize=8)
ax.set_title("P(menu | sex × age_10)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 5. 자주 가는 카페 브랜드 (SQ4 + visit_*) × 인구통계 → `prior_brand_by_demographic.csv`

In [ ]:
if sq4_cols:
    print("[SQ4] 1순위 분포:")
    print(survey_df[sq4_cols[0]].value_counts(dropna=False).head(15).to_string())

if visit_cols:
    print("\n[visit_*] 카페별 방문 비율:")
    print(survey_df[visit_cols].mean().sort_values(ascending=False).to_string())

In [ ]:
if visit_cols:
    brand_by_demo = (
        survey_df.groupby(["sex","age_10"])[visit_cols].mean().reset_index()
    )
    brand_by_demo.to_csv(OUTPUT_DIR / "prior_brand_by_demographic.csv", index=False, encoding="utf-8-sig")
    print("saved prior_brand_by_demographic.csv shape:", brand_by_demo.shape)
    brand_by_demo.head()

## 6. 시간대 (Q8) — `prior_time_by_demographic.csv`

In [ ]:
# 모든 카페에 걸친 시간대별 방문 응답률 평균
time_avg = {}
for t in times_q8:
    cols = [c for c in q8_cols if c.endswith(f"_Q8_{t}")]
    if cols:
        time_avg[t] = survey_df[cols].mean().mean()
time_avg = pd.Series(time_avg).sort_values(ascending=False)
print("[Q8] 시간대별 방문 응답률 (모든 카페 평균):")
print(time_avg.to_string())

fig, ax = plt.subplots(figsize=(8,4))
time_avg.plot(kind="bar", ax=ax, color="#34d399")
ax.set_xlabel("time period")
ax.set_ylabel("response rate (mean)")
ax.set_title("Q8 — 시간대별 방문 응답률 (모든 카페 평균)")
plt.tight_layout()
plt.show()

In [ ]:
# (sex × age_10) × time_period 평균
rows = []
for t in times_q8:
    cols = [c for c in q8_cols if c.endswith(f"_Q8_{t}")]
    if not cols:
        continue
    g = survey_df.groupby(["sex","age_10"])[cols].mean().mean(axis=1).reset_index(name="rate")
    g["time_period"] = t
    rows.append(g)
time_by_demo = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
time_by_demo.to_csv(OUTPUT_DIR / "prior_time_by_demographic.csv", index=False, encoding="utf-8-sig")
print("saved prior_time_by_demographic.csv shape:", time_by_demo.shape)
time_by_demo.head()

## 7. 메뉴 (Q9) × 시간대 (Q8) → `prior_menu_by_time.csv`

Q9는 `<카페>_Q9_<메뉴>` 형태. Q8는 `<카페>_Q8_<시간대>` 형태.
동일 응답자에서 Q9 메뉴와 Q8 시간대 둘 다 1인 경우의 빈도로 P(menu | time) 추정.

In [ ]:
# 응답자별로 (cafe, menu) flatten + (cafe, time) flatten 후 inner-cafe로 결합
rows = []
for menu in menus_q9:
    cols_m = [c for c in q9_cols if c.endswith(f"_Q9_{menu}")]
    if not cols_m:
        continue
    menu_signal = survey_df[cols_m].sum(axis=1) > 0  # 어떤 카페에서든 그 메뉴를 마심
    for t in times_q8:
        cols_t = [c for c in q8_cols if c.endswith(f"_Q8_{t}")]
        if not cols_t:
            continue
        time_signal = survey_df[cols_t].sum(axis=1) > 0
        joint = (menu_signal & time_signal).sum()
        time_only = time_signal.sum()
        prob = joint / time_only if time_only > 0 else 0.0
        rows.append({"menu": menu, "time_period": t, "P_menu_given_time": prob,
                     "joint_count": int(joint), "time_count": int(time_only)})
menu_by_time = pd.DataFrame(rows)
menu_by_time.to_csv(OUTPUT_DIR / "prior_menu_by_time.csv", index=False, encoding="utf-8-sig")
print("saved prior_menu_by_time.csv shape:", menu_by_time.shape)
menu_by_time.sort_values("P_menu_given_time", ascending=False).head(20)

## 8. rec_items.csv 보조 분석 (메뉴 메타 검증)

In [ ]:
print(items_df.shape)
print(items_df.columns.tolist())
items_df.head()

## 9. 외부 reference 검증 (Phase 1.2)

본 섹션은 OpenSurvey 분포가 실제 카페 시장과 어긋나지 않는지 정량 비교한다.
사용 가능한 reference:
- (필수) **노트북 01의 Bread Basket 시간대 분포** — 영국식이지만 카페 운영 시간 패턴 reference.
- (선택) **Coffee Sales Dataset** — 미국 카페 시간대 분포.
- (정성) NIQ Korea / Simon-Kucher PDF — 직접 수치 비교는 어려움, 정성 메모만.

Bread Basket 정제본이 Drive/로컬에 있으면 자동 로드. 없으면 검증 섹션은 skip.

In [ ]:
validation_lines = ["# Validation Report — OpenSurvey vs External References\n\n"]

# Bread Basket 후보 경로 (어디 두셨든 자동 탐색)
bread_candidates = [
    OUTPUT_DIR / "transactions_clean.csv",
    DATA_DIR / "output" / "transactions_clean.csv",
    DATA_DIR / "bread basket.csv",
    DATA_DIR / "bread_basket" / "bread basket.csv",
    DATA_DIR / "raw" / "bread basket.csv",
    DATA_DIR / "raw" / "bread_basket" / "bread basket.csv",
    BASE_DIR / "bread basket.csv" if "BASE_DIR" in dir() else None,
]
bread_candidates = [p for p in bread_candidates if p is not None]
BREAD_PATH = next((p for p in bread_candidates if p.exists()), None)
print("BREAD_PATH:", BREAD_PATH)

if BREAD_PATH is not None:
    bread = pd.read_csv(BREAD_PATH)
    # 두 형태 모두 처리: 노트북 01 정제본(컬럼 hour) vs 원본 raw(date_time 파싱 필요)
    if "hour" not in bread.columns:
        bread["datetime"] = pd.to_datetime(bread["date_time"], errors="coerce", dayfirst=True)
        bread["hour"] = bread["datetime"].dt.hour
    tx_col = "Transaction" if "Transaction" in bread.columns else bread.columns[0]
    bread_hour = bread.dropna(subset=["hour"]).drop_duplicates(tx_col).groupby("hour").size()
    bread_hour = bread_hour / bread_hour.sum()

    print("[Bread Basket — hour distribution]")
    print(bread_hour.round(3).to_string())
    fig, ax = plt.subplots(figsize=(10,3))
    bread_hour.plot(kind="bar", ax=ax, color="#f59e0b")
    ax.set_title("Reference: Bread Basket — orders by hour (normalized)")
    plt.tight_layout(); plt.show()
    validation_lines.append(f"## Bread Basket reference\n- Peak hour: {int(bread_hour.idxmax())}h\n- Source: `{BREAD_PATH}`\n\n")
else:
    print("[skip] Bread Basket 파일 없음 — 후보 경로:")
    for p in bread_candidates:
        print("  -", p)
    validation_lines.append("## Bread Basket reference: SKIPPED (file missing)\n\n")

In [ ]:
# Coffee Sales 후보 경로 (폴더 구조 차이를 자동 흡수)
coffee_candidates = [
    DATA_DIR / "coffee sales dataset" / "Coffe_sales.csv",
    DATA_DIR / "Coffe_sales.csv",
    DATA_DIR / "coffee_sales" / "Coffe_sales.csv",
    DATA_DIR / "raw" / "coffee sales dataset" / "Coffe_sales.csv",
    DATA_DIR / "raw" / "Coffe_sales.csv",
    BASE_DIR / "coffee sales dataset" / "Coffe_sales.csv" if "BASE_DIR" in dir() else None,
    BASE_DIR / "Coffe_sales.csv" if "BASE_DIR" in dir() else None,
]
coffee_candidates = [p for p in coffee_candidates if p is not None]
COFFEE_PATH = next((p for p in coffee_candidates if p.exists()), None)
print("COFFEE_PATH:", COFFEE_PATH)

if COFFEE_PATH is not None:
    coffee = read_csv_smart(COFFEE_PATH)
    # hour 컬럼 결정
    if "hour_of_day" in coffee.columns:
        coffee["hour"] = pd.to_numeric(coffee["hour_of_day"], errors="coerce")
    elif "hour" in coffee.columns:
        coffee["hour"] = pd.to_numeric(coffee["hour"], errors="coerce")
    else:
        coffee["hour"] = pd.to_datetime(coffee.get("Time", coffee.get("datetime", None)), errors="coerce").dt.hour
    h = coffee["hour"].dropna().astype(int).value_counts().sort_index()
    h = h / h.sum()
    print("[Coffee Sales — hour distribution]")
    print(h.round(3).to_string())
    fig, ax = plt.subplots(figsize=(10,3))
    h.plot(kind="bar", ax=ax, color="#0ea5e9")
    ax.set_title("Reference: Coffee Sales Dataset — orders by hour (normalized)")
    plt.tight_layout(); plt.show()
    validation_lines.append(f"## Coffee Sales reference\n- Peak hour: {int(h.idxmax())}h\n- Source: `{COFFEE_PATH}`\n\n")
else:
    print("[skip] Coffee Sales 파일 없음 — 후보 경로:")
    for p in coffee_candidates:
        print("  -", p)
    validation_lines.append("## Coffee Sales reference: SKIPPED\n\n")

In [ ]:
validation_lines.append("\n## Summary\n")
validation_lines.append("- OpenSurvey 인구통계: 자기검증 (표집 그대로 사용)\n")
validation_lines.append("- 시간대 분포: Bread Basket / Coffee Sales 두 reference의 피크 시간이 OpenSurvey의 'Q8 점심/오전' 응답률 피크와 정성적으로 일치하는지 확인\n")
validation_lines.append("- 메뉴 카테고리 비중 / 효과 크기: 자료조사 PDF에 명시된 수치와 합성 단계(노트북 03)에서 비교\n")

report_path = OUTPUT_DIR / "validation_report.md"
report_path.write_text("".join(validation_lines), encoding="utf-8")
print("saved:", report_path)
print("".join(validation_lines))

## 10. 다운로드

직접 업로드 모드일 때 산출물을 로컬로 내려받는다.

In [ ]:
if not USE_GDRIVE:
    try:
        from google.colab import files
        for name in ["prior_demographics.csv",
                     "prior_menu_by_demographic.csv",
                     "prior_brand_by_demographic.csv",
                     "prior_time_by_demographic.csv",
                     "prior_menu_by_time.csv",
                     "validation_report.md"]:
            p = OUTPUT_DIR / name
            if p.exists():
                files.download(str(p))
    except Exception as e:
        print("download skipped (non-Colab env):", e)

## 11. 다음 단계

1. 본 노트북 산출 5종 prior CSV + validation_report 검토.
2. 검증에서 어긋난 부분이 있으면 다음 노트북(`03_build_and_validate_synthetic.ipynb`)의 합성 SPEC에 보정 메모로 반영.
3. 노트북 03에서 자체 합성 실행 → 검증 → Phase 2(모델 학습)로 진입.